<a href="https://colab.research.google.com/github/missstechie/Intelligent-CC-Suggestion-Tool/blob/main/Intelligent_CC_Suggestion_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Intelligent Closed Caption (CC) Suggestion Tool (MVP)

This project demonstrates a simplified approach to generating closed caption (CC) suggestions from video/audio input.

Instead of manual annotation, this prototype detects significant audio events (based on intensity) and generates timestamped captions.

### Goal:
Reduce manual effort in identifying meaningful non-speech audio events.


In [1]:
!pip install moviepy librosa


In [13]:
video_url = "https://www.youtube.com/watch?v=mbqCXpmo15A&t=2s" #@param {type:"string"}
# Use yt-dlp to download the video
!yt-dlp -o "%(id)s.%(ext)s" "{video_url}"
# yt-dlp will automatically extract the video ID and extension
# We need to find the actual filename that yt-dlp downloaded
import glob
# yt-dlp downloaded a webm file in this case after merging
video_file = glob.glob('*.webm')[0]

[youtube] Extracting URL: https://www.youtube.com/watch?v=mbqCXpmo15A&t=2s
[youtube] mbqCXpmo15A: Downloading webpage
[youtube] mbqCXpmo15A: Downloading android vr player API JSON
[info] mbqCXpmo15A: Downloading 1 format(s): 401+251
[download] mbqCXpmo15A.webm has already been downloaded


In [11]:
!pip install yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 30.6 MB/s eta 0:00:00


In [14]:
from moviepy.editor import VideoFileClip

# video_file = list(uploaded.keys())[0] # Modified to use the downloaded video_file

video = VideoFileClip(video_file)
audio = video.audio
audio.write_audiofile("audio.wav")

MoviePy - Writing audio in audio.wav


MoviePy - Done.


In [15]:
import librosa
import numpy as np

y, sr = librosa.load("audio.wav")

energy = np.abs(y)
threshold = np.mean(energy) * 3

event_times = []

for i, e in enumerate(energy):
   if e > threshold:
       event_times.append(i / sr)


In [16]:
filtered_events = []

for t in event_times:
   if not filtered_events or t - filtered_events[-1] > 2:
       filtered_events.append(t)

In [17]:
def format_time(t):
   mins = int(t // 60)
   secs = int(t % 60)
   return f"00:{mins:02d}:{secs:02d},000"

with open("output.srt", "w") as f:
   for i, t in enumerate(filtered_events[:10]):
       f.write(f"{i+1}\n")
       f.write(f"{format_time(t)} --> {format_time(t+2)}\n")
       f.write("[Loud Sound Detected]\n\n")

print("SRT file generated!")

SRT file generated!


##  Limitations
- Uses simple energy-based detection (not semantic audio classification)
- No visual reaction detection yet
- May detect irrelevant loud sounds

##  Future Work
- Integrate sound classification models (e.g., YAMNet)
- Add visual reaction detection using OpenCV
- Improve filtering logic
